In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pcsv
import csv
import re
import sys
import numpy as np
import pyreadstat

In [2]:
# set paths 
fts_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.fts"
dat_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.dat"
out_path = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/dat_test_nopandas.parquet"
sas_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/2000/denominator/dnm2000.sas7bdat"
sas_out = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/sas.parquet"

In [3]:
# parse fts file to create dictionary of schema
def read_fts(fts_file):
    """
    Reads a .fts file and extracts column metadata into a structured dictionary.
    
    Args:
        fts_file (str): Path to the .fts file.
    
    Returns:
        dict: Parsed data with column headers as keys and row values as lists.
    """
    parsing = False  # Flag to start parsing after ----
    headers = [] 
    data_dict = {}
    header_lines = [] # fts headers
    column_widths = []  # width of fts columns
    
    with open(fts_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Find the header lines and the start of the table
    for i, line in enumerate(lines):
        line = line.rstrip()
        if '----' in line:
            parsing = True  # Start processing after this line
            if i > 2:
                header_lines = [l.rstrip() for l in lines[i-3:i]]  # headers are in the 3 lines before the '---'
            
            # Determine column widths of fts file by measuring dashes
            column_widths = [len(match.group()) for match in re.finditer(r'-+', line)]
            break
    
    # Extract column headers from three stacked lines
    start = 0
    for width in column_widths:
        column_header = ' '.join(line[start:start+width].strip() for line in header_lines).strip()
        headers.append(column_header)
        start += width + 1  # Move to next column start position
    
    # Initialize data dictionary with headers
    for header in headers:
        data_dict[header] = []
    
    # Process the data rows using fixed-width slicing based on column width
    for line in lines[i+1:]:  # Start from the first row after ----
        line = line.rstrip()
        if not line:
            continue
        
        # Stop parsing if the line contains "Note:" ### change this to make more robust 
        if "Note:" in line:
            break
        
        start = 0
        row_values = []
        for width in column_widths:
            row_values.append(line[start:start+width].strip())
            start += width + 1
        
        if len(row_values) == len(headers):
            for header, value in zip(headers, row_values):
                data_dict[header].append(value)
    
    return data_dict


In [14]:
data_dict = read_fts(fts_path)
df = pd.DataFrame(data_dict)
df

,Fld. No.,Field Long Name,Field Short Name,Field Generic Format,Start Col.,Fld. Len. w.d,Field Description
0,1,BENE_ID,BENE_ID,CHAR,1,15,Encrypted 723 Beneficiary ID (Unique Key)
1,2,BENE_ENROLLMT_REF_YR,RFRNC_YR,NUM,16,4,Beneficiary Enrollment Reference Year
2,3,FIVE_PERCENT_FLAG,FIVEPCT,CHAR,20,1,Strict 5% Flag
3,4,ENHANCED_FIVE_PERCENT_FLAG,EFIVEPCT,CHAR,21,1,Enhanced 5% Flag
4,5,COVSTART,COVSTART,DATE,22,8,Medicare Coverage Start Date
5,6,CRNT_BIC_CD,CRNT_BIC,CHAR,30,2,Beneficiary Identification Code
6,7,STATE_CODE,STATE_CD,CHAR,32,2,State Code
7,8,BENE_COUNTY_CD,CNTY_CD,CHAR,34,3,County Code
8,9,BENE_ZIP_CD,BENE_ZIP,CHAR,37,9,Zip Code of Residence
9,10,BENE_AGE_AT_END_REF_YR,AGE,NUM,46,3,Age at End of Reference Year


First, attempt to simply parse dat file as csv using the schema from fts file

In [7]:
def parse_dat_csv(dat_file, data_dict, output_csv, parquet=False, max_rows=100):
    """
    Parses a .dat file using the column widths and headers from data_dict and saves as CSV.
    Processes only the first `max_rows` rows.

    Args:
        dat_file (str): Path to the .dat file.
        data_dict (dict): Dictionary containing column metadata from read_fts.
        output_csv (str): Path to save the output CSV file.
        parquet (bool, optional): If True, also saves as Parquet. Default is False.
        max_rows (int, optional): Maximum number of rows to process. Default is 100.
    """
    keys = list(data_dict.keys())
    
    headers = data_dict[keys[2]]  # Extract headers from the 3rd key 
    column_widths = [int(width) for width in data_dict[keys[5]]]  # Extract column widths from the 6th key
    
    with open(dat_file, 'r', encoding='utf-8') as f, open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)  # Write headers
        
        for i, line in enumerate(f):
            if i >= max_rows:
                break  # Stop after processing max_rows
            
            line = line.rstrip()
            start = 0
            row_values = []
            
            for width in column_widths:
                row_values.append(line[start:start+width].strip())
                start += width
            
            writer.writerow(row_values)



In [8]:
parse_dat_csv(dat_path,data_dict,out_path)
import pandas as pd 
pd.read_csv(out_path)


,BENE_ID,RFRNC_YR,FIVEPCT,EFIVEPCT,COVSTART,CRNT_BIC,STATE_CD,CNTY_CD,BENE_ZIP,AGE,...,HMOIND03,HMOIND04,HMOIND05,HMOIND06,HMOIND07,HMOIND08,HMOIND09,HMOIND10,HMOIND11,HMOIND12
0,llllllllllllllS,2012,NaN,NaN,19810801,A,30,60,33032410,96,...,0,0,0,0,0,0,0,0,0,0
1,lllllllllllll0l,2012,NaN,NaN,19820601,D,10,100,341107051,94,...,0,0,0,0,0,0,0,0,0,0
2,lllllllllllll07,2012,NaN,NaN,19810501,A,30,60,33013228,96,...,0,0,0,0,0,0,0,0,0,0
3,lllllllllllll0S,2012,NaN,NaN,19850301,D,30,60,33017810,92,...,0,0,0,0,0,0,0,0,0,0
4,lllllllllllllU0,2012,NaN,NaN,19751101,C1,30,60,33013549,57,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,llllllllllllO0O,2012,NaN,NaN,19990601,A,47,20,58191604,78,...,C,C,C,C,C,C,C,C,C,C
96,llllllllllllOU4,2012,NaN,NaN,19871101,D,22,170,15341200,90,...,0,0,0,0,0,0,0,0,0,0
97,llllllllllllOUo,2012,NaN,NaN,19850901,D,30,50,30860000,92,...,0,0,0,0,0,0,0,0,0,0
98,llllllllllllOXX,2012,NaN,NaN,19810701,A,30,70,30382035,96,...,0,0,0,0,0,0,0,0,0,0


Parsing appears to be successful. Now will handle datatypes and converting to parquet.

In [9]:
def change_dftypes(df, type_dict, verbose=False):
    """
    Converts column data types based on provided type mapping.
    
    Args:
        df (pd.DataFrame): Input DataFrame.
        type_dict (dict): Dictionary mapping column names to desired data types.
        verbose (bool): Whether to print type conversion messages.

    Returns:
        pd.DataFrame: DataFrame with updated types.
    """
    for col in df.columns:
        
        dtype = type_dict[col]

        if dtype == 'NUM':
            if verbose: print(f"{col}: CHAR --> NUM")
            df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')

        elif dtype == 'DATE':
            if verbose: print(f"{col}: CHAR --> DATE")
            df[col] = pd.to_datetime(df[col].str.strip(), errors='coerce')

        else:  # Default to string (CHAR)
            df[col] = df[col].str.strip().replace('', np.nan)
            if verbose: print(f"{col}: CHAR")

    return df


def parse_dat_parq_csv(dat_file, data_dict, output_csv, parquet=False, max_rows=20, verbose=False):
    """
    Parses a .dat file using the column widths and headers from data_dict, converts data types, and saves as CSV.
    
    Args:
        dat_file (str): Path to the .dat file.
        data_dict (dict): Dictionary containing column metadata from read_fts.
        output_csv (str): Path to save the output CSV file.
        parquet (bool): If True, also saves as Parquet. Default is False.
        max_rows (int): Maximum number of rows to process. Default is 100.
        verbose (bool): Whether to print debugging info.

    Returns:
        pd.DataFrame: Processed DataFrame.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    keys = list(data_dict.keys())

    headers = data_dict[keys[2]]  # Extract headers from the 3rd key
    column_widths = [int(width) for width in data_dict[keys[5]]]  # Extract column widths from the 6th key
    data_types = dict(zip(headers, data_dict[keys[3]]))  # Extract data types into a dictionary

    data = {header: [] for header in headers}  # Initialize storage

    # Read and parse .dat file
    with open(dat_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_rows:
                break  # Stop after max_rows
            
            line = line.rstrip()
            start = 0
            row_values = []
            
            for header, width in zip(headers, column_widths):
                row_values.append(line[start:start+width].strip())
                start += width

            for header, value in zip(headers, row_values):
                data[header].append(value)

    df = pd.DataFrame(data)

    # Apply type casting
    df = change_dftypes(df, data_types, verbose=verbose)

    df.to_csv(output_csv, index=False)
    
    if parquet:
        table = pa.Table.from_pandas(df)
        pq.write_table(table, output_csv.replace('.csv', '.parquet'))

    return df


In [10]:
parse_dat_parq_csv(dat_path, data_dict, out_path, parquet=False, max_rows=1000, verbose=False)

,BENE_ID,RFRNC_YR,FIVEPCT,EFIVEPCT,COVSTART,CRNT_BIC,STATE_CD,CNTY_CD,BENE_ZIP,AGE,...,HMOIND03,HMOIND04,HMOIND05,HMOIND06,HMOIND07,HMOIND08,HMOIND09,HMOIND10,HMOIND11,HMOIND12
0,llllllllllllllS,2012,NaN,NaN,1981-08-01,A,30,060,033032410,96,...,0,0,0,0,0,0,0,0,0,0
1,lllllllllllll0l,2012,NaN,NaN,1982-06-01,D,10,100,341107051,94,...,0,0,0,0,0,0,0,0,0,0
2,lllllllllllll07,2012,NaN,NaN,1981-05-01,A,30,060,033013228,96,...,0,0,0,0,0,0,0,0,0,0
3,lllllllllllll0S,2012,NaN,NaN,1985-03-01,D,30,060,033017810,92,...,0,0,0,0,0,0,0,0,0,0
4,lllllllllllllU0,2012,NaN,NaN,1975-11-01,C1,30,060,033013549,57,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,lllllllllllO4S4,2012,NaN,NaN,1980-12-01,A,30,020,034690554,97,...,0,0,0,0,0,0,0,0,0,0
996,lllllllllllO4SS,2012,NaN,NaN,1986-02-01,D,07,060,060294245,91,...,0,0,0,0,0,0,0,0,0,0
997,lllllllllllO4S8,2012,NaN,NaN,1979-10-01,A,50,130,985410705,97,...,0,0,0,0,0,0,0,0,0,0
998,lllllllllllO484,2012,NaN,NaN,1980-09-01,A,10,570,342752374,97,...,0,0,0,0,0,0,0,0,0,0


In [60]:
table = pq.read_table("/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/dat_test.parquet")
csv_table = pd.read_csv("/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/dat_test.csv")
csv_table

,BENE_ID,RFRNC_YR,FIVEPCT,EFIVEPCT,COVSTART,CRNT_BIC,STATE_CD,CNTY_CD,BENE_ZIP,AGE,...,HMOIND03,HMOIND04,HMOIND05,HMOIND06,HMOIND07,HMOIND08,HMOIND09,HMOIND10,HMOIND11,HMOIND12
0,llllllllllllllS,2012,NaN,NaN,1981-08-01,A,30.0,60,33032410,96,...,0,0,0,0,0,0,0,0,0,0
1,lllllllllllll0l,2012,NaN,NaN,1982-06-01,D,10.0,100,341107051,94,...,0,0,0,0,0,0,0,0,0,0
2,lllllllllllll07,2012,NaN,NaN,1981-05-01,A,30.0,60,33013228,96,...,0,0,0,0,0,0,0,0,0,0
3,lllllllllllll0S,2012,NaN,NaN,1985-03-01,D,30.0,60,33017810,92,...,0,0,0,0,0,0,0,0,0,0
4,lllllllllllllU0,2012,NaN,NaN,1975-11-01,C1,30.0,60,33013549,57,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,llllllll0S0o848,2012,NaN,NaN,2000-07-01,A,10.0,510,337642779,77,...,0,0,0,0,0,0,0,0,0,0
999996,llllllll0S0o8ol,2012,NaN,NaN,1998-01-01,A,10.0,630,321747503,79,...,0,0,0,0,0,0,0,0,0,0
999997,llllllll0S0o8o0,2012,NaN,NaN,2000-07-01,A,22.0,20,27660650,77,...,C,C,C,C,C,C,C,C,C,C
999998,llllllll0S0o8o4,2012,NaN,NaN,1999-09-01,A,22.0,80,10829496,78,...,C,C,C,C,C,C,C,C,C,C


In [61]:
print(table)

pyarrow.Table
BENE_ID: string
RFRNC_YR: int64
FIVEPCT: string
EFIVEPCT: string
COVSTART: timestamp[ns]
CRNT_BIC: string
STATE_CD: string
CNTY_CD: string
BENE_ZIP: string
AGE: int64
BENE_DOB: timestamp[ns]
V_DOD_SW: string
DEATH_DT: timestamp[ns]
NDI_DEATH_DT: timestamp[ns]
SEX: string
RACE: string
RTI_RACE_CD: string
OREC: string
CREC: string
ESRD_IND: string
MS_CD: string
A_TRM_CD: string
B_TRM_CD: string
A_MO_CNT: int64
B_MO_CNT: int64
BUYIN_MO: int64
HMO_MO: int64
BUYIN01: string
BUYIN02: string
BUYIN03: string
BUYIN04: string
BUYIN05: string
BUYIN06: string
BUYIN07: string
BUYIN08: string
BUYIN09: string
BUYIN10: string
BUYIN11: string
BUYIN12: string
HMOIND01: string
HMOIND02: string
HMOIND03: string
HMOIND04: string
HMOIND05: string
HMOIND06: string
HMOIND07: string
HMOIND08: string
HMOIND09: string
HMOIND10: string
HMOIND11: string
HMOIND12: string
----
BENE_ID: [["llllllllllllllS","lllllllllllll0l","lllllllllllll07","lllllllllllll0S","lllllllllllllU0",...,"lllllllllU4070l","lll

Seems like this methodology works, but it is reliant on pandas. Try to develop without dependence on pandas for more efficient processing 
### No pandas parsing

something weird going on with the dates...

In [ ]:
# reformat dates to work w numpy 
# can we read a fwf file efficiently without having to parse line by line? 
# retain csv files (need to update numpy function to do this)
# before 2011 we may not have resdac raw files? 
## parse .sas7bdat files for before 2011 
# look @ this: https://github.com/NSAPH-Data-Processing/legacy_mbsf_mortality_denom/blob/main/src/denom.py
# look @ 

# would be nice
## parsing by column 
## read file by chunk, in a byte sequence

### run on the 2015 data once complete

Using only pyarrow and numpy!!!

Parse dat file column wise instead of rowwise 

In [39]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pc
import numpy as np
from datetime import datetime

def change_dftypes_pyarrow(data, type_dict, verbose=False):
    """
    Converts column data types based on provided type mapping using PyArrow.
    """
    for col, values in data.items():
        dtype = type_dict.get(col, "CHAR")
        
        if dtype == 'NUM':
            if verbose: print(f"{col}: CHAR --> NUM")
            data[col] = pa.array([float(v) if v.replace('.', '', 1).isdigit() else None for v in values], type=pa.float64())
        
        elif dtype == 'DATE':
            if verbose: print(f"{col}: CHAR --> DATE")
            data[col] = pa.array([datetime.strptime(v, "%Y%m%d") if v else None for v in values], type=pa.date64())
        
        else:  # Default to string (CHAR)
            data[col] = pa.array([v.strip() if v.strip() else None for v in values], type=pa.string())
            if verbose: print(f"{col}: CHAR")
    
    return data

def parse_dat_parq_csv_pyarrow(dat_file, data_dict, output_csv, parquet=False, max_rows=20, verbose=False):
    """
    Parses a .dat file column-wise using the column widths and headers from data_dict, converts data types, and saves as CSV or Parquet.
    """
    keys = list(data_dict.keys())
    headers = data_dict[keys[2]]  # Extract headers from the 3rd key
    column_widths = [int(width) for width in data_dict[keys[5]]]  # Extract column widths from the 6th key
    data_types = dict(zip(headers, data_dict[keys[3]]))  # Extract data types into a dictionary
    
    data = {header: [] for header in headers}  # Initialize storage
    
    # Read and parse .dat file column by column
    with open(dat_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:max_rows]  # Read all lines up to max_rows
        
        for col_idx, (header, width) in enumerate(zip(headers, column_widths)):
            if verbose: print(f"Processing column: {header}")
            
            col_values = [line[sum(column_widths[:col_idx]):sum(column_widths[:col_idx]) + width].strip() for line in lines]
            data[header] = col_values
    
    # Apply type casting
    data = change_dftypes_pyarrow(data, data_types, verbose=verbose)
    
    # Convert to PyArrow table
    table = pa.table(data)
    
    # Write CSV
    pc.write_csv(table, output_csv)
    
    # Write Parquet if required
    if parquet:
        pq.write_table(table, output_csv.replace('.csv', '.parquet'))
    
    return table


In [40]:
parse_dat_parq_csv_pyarrow(dat_path, data_dict, out_path, parquet=True, max_rows=20, verbose=False)

pyarrow.Table
BENE_ID: string
RFRNC_YR: double
FIVEPCT: string
EFIVEPCT: string
COVSTART: date64[ms]
CRNT_BIC: string
STATE_CD: string
CNTY_CD: string
BENE_ZIP: string
AGE: double
BENE_DOB: date64[ms]
V_DOD_SW: string
DEATH_DT: date64[ms]
NDI_DEATH_DT: date64[ms]
SEX: string
RACE: string
RTI_RACE_CD: string
OREC: string
CREC: string
ESRD_IND: string
MS_CD: string
A_TRM_CD: string
B_TRM_CD: string
A_MO_CNT: double
B_MO_CNT: double
BUYIN_MO: double
HMO_MO: double
BUYIN01: string
BUYIN02: string
BUYIN03: string
BUYIN04: string
BUYIN05: string
BUYIN06: string
BUYIN07: string
BUYIN08: string
BUYIN09: string
BUYIN10: string
BUYIN11: string
BUYIN12: string
HMOIND01: string
HMOIND02: string
HMOIND03: string
HMOIND04: string
HMOIND05: string
HMOIND06: string
HMOIND07: string
HMOIND08: string
HMOIND09: string
HMOIND10: string
HMOIND11: string
HMOIND12: string
----
BENE_ID: [["llllllllllllllS","lllllllllllll0l","lllllllllllll07","lllllllllllll0S","lllllllllllllU0",...,"llllllllllll00l","lllllllll

process sas7bdat file

In [ ]:
import pyreadstat
import pyarrow as pa
from datetime import datetime

def change_dftypes_pyarrow(data, type_dict, verbose=False):
    """
    Converts column data types based on provided type mapping using PyArrow.
    """
    for col, values in data.items():
        dtype = type_dict.get(col, "CHAR")
        
        if dtype == 'NUM':
            if verbose: print(f"{col}: CHAR --> NUM")
            data[col] = pa.array([float(v) if v.replace('.', '', 1).isdigit() else None for v in values], type=pa.float64())
        
        elif dtype == 'DATE':
            if verbose: print(f"{col}: CHAR --> DATE")
            data[col] = pa.array([datetime.strptime(v, "%Y%m%d") if v else None for v in values], type=pa.date64())
        
        else:  # Default to string (CHAR)
            data[col] = pa.array([v.strip() if v.strip() else None for v in values], type=pa.string())
            if verbose: print(f"{col}: CHAR")
    
    return data

def process_sas_columnwise(sas_file,output_csv, max_rows=20, parquet=True, verbose=False):
    """
    Process a SAS file column-wise and save it as CSV or Parquet without using pandas.
    """
    # Read the .sas7bdat file using pyreadstat
    df, meta = pyreadstat.read_sas7bdat(sas_file, row_offset=1, row_limit=20)  # Read a subset
    
    headers = meta.column_names  # Get column names
    column_types = meta.column_types  # Get column types (for type casting)
    
    # column_widths = [len(header) for header in headers]  # You can set widths based on your data_dict if needed
    data_types = dict(zip(headers, column_types))  # Map column names to types

    # Initialize the data dictionary
    data = {header: [] for header in headers}

    # Process the SAS data column-wise
    for col in headers:
        values = df[col].tolist()  # Get the column values
        data[col] = values

    # Apply type casting
    data = change_dftypes_pyarrow(data, data_types, verbose=verbose)
    
    # Convert to PyArrow table
    table = pa.table(data)
    
    # Write CSV
    pc.write_csv(table, output_csv)
    
    # Write Parquet if required
    if parquet:
        pq.write_table(table, output_csv.replace(".csv", ".parquet"))
    
    return table

# Example usage:
process_sas_columnwise(sas_path,sas_out, max_rows=20, parquet=True)


Editing function to run slurm job to chunk the data rowwise